In [1]:
import pandas as pd
import numpy as np
import os

# Setting the file path
RAW_FILE_PATH = 'household_power_consumption.csv'

# Defining the inspection function
def inspect_raw_source(file_path):
    print("=" * 60)
    print("🔍 STEP 1.1: SOURCE VERIFICATION & COLUMN INSPECTION")
    print("=" * 60)
    
    # Checking file existence and size
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Raw dataset not found at {file_path}. Please place the CSV in the root directory.")
    
    file_size_mb = os.path.getsize(file_path) / (1024 * 1024) 
    print(f"📁 File Name: {file_path}")
    print(f"📊 File Size: {file_size_mb:.2f} MB")
    
    # Inspecting raw lines without loading the whole file
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        header = f.readline().strip().split(';')
        first_row = f.readline().strip().split(';')
        
    print(f"\n📋 Detected Columns ({len(header)} total):")
    for i, col in enumerate(header, 1):
        print(f"  {i}. {col}")

    # Inspecting sample values
    print(f"\n🔬 Sample Raw Row:")
    for col, val in zip(header, first_row):
        print(f"  {col}: {val}")

# Execute the function
inspect_raw_source(RAW_FILE_PATH)

🔍 STEP 1.1: SOURCE VERIFICATION & COLUMN INSPECTION
📁 File Name: household_power_consumption.csv
📊 File Size: 47.03 MB

📋 Detected Columns (1 total):
  1. Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3

🔬 Sample Raw Row:
  Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3: 16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0,1,17


In [6]:
import pandas as pd
import numpy as np

def parse_and_construct_timestamps(file_path):
    print("=" * 60)
    print("⚙️ STEP 1.2: PARSING, TYPES & TIMESTAMP CONSTRUCTION")
    print("=" * 60)
    
    # 1. Load data with automatic delimiter detection or explicit fallback
    print("📂 Loading dataset into memory with flexible delimiter detection...")
    try:
        # Try semicolon first (original UCI dataset format)
        df = pd.read_csv(file_path, sep=';', na_values=['?'], low_memory=False)
        if len(df.columns) <= 1:
            raise ValueError("Semicolon separator resulted in 1 column. Trying comma...")
    except Exception:
        # Fallback to comma or auto-engine
        df = pd.read_csv(file_path, sep=None, engine='python', na_values=['?'])
        
    print(f"  • Initial DataFrame Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print(f"  • Detected Columns: {list(df.columns)}")
    
    # 2. Combine Date and Time columns into a unified datetime index
    print("⏳ Parsing and merging 'Date' and 'Time' into a datetime index...")
    df['DateTime'] = pd.to_datetime(
        df['Date'] + ' ' + df['Time'],
        format='%d/%m/%Y %H:%M:%S',
        errors='coerce'
    )
    
    # 3. Check for parsing failures
    invalid_timestamps = df['DateTime'].isnull().sum()
    print(f"  • Unparseable / Invalid Timestamps Dropped: {invalid_timestamps:,}")
    
    # Drop rows with invalid timestamps and set DateTime as index
    df = df.dropna(subset=['DateTime'])
    df = df.set_index('DateTime')
    
    # Drop redundant original string columns
    df = df.drop(columns=['Date', 'Time'])
    
    # 4. Explicitly convert all electrical measurement columns to numeric data types
    electrical_cols = [
        'Global_active_power', 'Global_reactive_power', 'Voltage',
        'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
    ]
    
    print("🔄 Converting electrical columns to explicit numeric types...")
    for col in electrical_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        
    print("\n📈 Final Parsed Data Types & Memory Profile:")
    print(df.dtypes)
    print(f"\n📊 Final Clean Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
    
    return df

# Execute Step 1.2 parsing module
df_parsed = parse_and_construct_timestamps('household_power_consumption.csv')

⚙️ STEP 1.2: PARSING, TYPES & TIMESTAMP CONSTRUCTION
📂 Loading dataset into memory with flexible delimiter detection...
  • Initial DataFrame Shape: 1,048,575 rows, 9 columns
  • Detected Columns: ['Date', 'Time', 'Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
⏳ Parsing and merging 'Date' and 'Time' into a datetime index...
  • Unparseable / Invalid Timestamps Dropped: 0
🔄 Converting electrical columns to explicit numeric types...

📈 Final Parsed Data Types & Memory Profile:
Global_active_power      float64
Global_reactive_power    float64
Voltage                  float64
Global_intensity         float64
Sub_metering_1           float64
Sub_metering_2           float64
Sub_metering_3           float64
dtype: object

📊 Final Clean Shape: 1,048,575 rows, 7 columns


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- 1.2: PARSING & TIMESTAMP CONSTRUCTION ---
def parse_and_construct_timestamps(file_path):
    print("=" * 60)
    print("⚙️ STEP 1.2: PARSING, TYPES & TIMESTAMP CONSTRUCTION")
    print("=" * 60)
    
    try:
        df = pd.read_csv(file_path, sep=';', na_values=['?'], low_memory=False)
        if len(df.columns) <= 1:
            raise ValueError()
    except Exception:
        df = pd.read_csv(file_path, sep=None, engine='python', na_values=['?'])
        
    df['DateTime'] = pd.to_datetime(
        df['Date'] + ' ' + df['Time'],
        format='%d/%m/%Y %H:%M:%S',
        errors='coerce'
    )
    
    df = df.dropna(subset=['DateTime'])
    df = df.set_index('DateTime')
    df = df.drop(columns=['Date', 'Time'])
    
    electrical_cols = [
        'Global_active_power', 'Global_reactive_power', 'Voltage',
        'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
    ]
    
    for col in electrical_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
    return df

# Initialize df_parsed
df_parsed = parse_and_construct_timestamps('household_power_consumption.csv')


# --- 1.3: MISSINGNESS ANALYSIS & TREATMENT ---
def analyze_and_treat_missingness(df):
    print("=" * 60)
    print("🔍 PHASE 1.3: MISSINGNESS ANALYSIS & TREATMENT")
    print("=" * 60)
    
    missing_counts = df.isnull().sum()
    missing_pct = (missing_counts / len(df)) * 100
    missing_df = pd.DataFrame({'Null Count': missing_counts, 'Missing Percentage (%)': missing_pct})
    print("\n📊 Null Counts & Percentages by Column:")
    print(missing_df[missing_df['Null Count'] > 0])
    
    is_null = df['Global_active_power'].isnull()
    gap_groups = (is_null != is_null.shift()).cumsum()
    gap_lengths = is_null.groupby(gap_groups).sum()
    valid_gaps = gap_lengths[gap_lengths > 0]
    
    print(f"\n⚡ Outage & Gap Statistics:")
    print(f"  • Total missing blocks detected: {len(valid_gaps):,}")
    print(f"  • Max contiguous gap duration: {valid_gaps.max()} minutes ({valid_gaps.max()/60:.1f} hours)")
    print(f"  • Short gaps (<= 60 mins): {len(valid_gaps[valid_gaps <= 60]):,}")
    print(f"  • Long outages (> 60 mins): {len(valid_gaps[valid_gaps > 60]):,}")
    
    df['Hour'] = df.index.hour
    df['Month'] = df.index.month
    hourly_missing = df[df['Global_active_power'].isnull()].groupby('Hour').size()
    print("\n⏰ Missingness Distribution by Hour of Day (Top 3 hours):")
    print(hourly_missing.nlargest(3))
    
    print("\n🛠️ Executing Treatment Policy...")
    df['GAP_Indicator'] = df['Global_active_power'].isnull().astype(int)
    
    df['Global_active_power_imputed'] = df['Global_active_power'].interpolate(
        method='linear', limit=60, limit_direction='both'
    )
    
    print(f"  • Imputed short gaps (<= 60 min limit). Remaining NaNs: {df['Global_active_power_imputed'].isnull().sum():,}")
    
    return df

# Execute Phase 1.3
df_analyzed = analyze_and_treat_missingness(df_parsed)

⚙️ STEP 1.2: PARSING, TYPES & TIMESTAMP CONSTRUCTION
🔍 PHASE 1.3: MISSINGNESS ANALYSIS & TREATMENT

📊 Null Counts & Percentages by Column:
                       Null Count  Missing Percentage (%)
Global_active_power          4069                 0.38805
Global_reactive_power        4069                 0.38805
Voltage                      4069                 0.38805
Global_intensity             4069                 0.38805
Sub_metering_1               4069                 0.38805
Sub_metering_2               4069                 0.38805
Sub_metering_3               4069                 0.38805

⚡ Outage & Gap Statistics:
  • Total missing blocks detected: 39
  • Max contiguous gap duration: 3723 minutes (62.0 hours)
  • Short gaps (<= 60 mins): 36
  • Long outages (> 60 mins): 3

⏰ Missingness Distribution by Hour of Day (Top 3 hours):
Hour
11    252
10    228
18    208
dtype: int64

🛠️ Executing Treatment Policy...
  • Imputed short gaps (<= 60 min limit). Remaining NaNs: 3,603


In [ ]:
import pandas as pd
import numpy as np

